# 04 — Systematic Clustering Experiments (All-Images Merged PCA)

Loops over **2 distance metrics × 2 k values = 4 combinations** using the
merged all-5-images PCA matrix (`pca_all_images_merged.csv`, 416 × 550 features).

| Distance | k values |
|---|---|
| `euclidean` | 3, 4 |
| `robust_mahalanobis` | 3, 4 |

**Evaluation:** % healthy spread · Silhouette · ARI · NMI  
**Output:** `results_all_images_merged.csv`

## Step 1 — Install dependencies

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-600:] if r.stderr else "(no stderr)")
    return r.returncode

# --no-deps skips scikit-learn-extra (requires MSVC to compile on Windows)
# db_robust_clust works fine without it at runtime
run_pip("--no-deps", "db-robust-clust==0.1.10")
for pkg in ["kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    db-robust-clust==0.1.10
  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Step 2 — Imports

In [2]:
import os
import numpy as np
import pandas as pd

import kmedoids
from db_robust_clust.models import SampleDistClustering
from robust_mixed_dist.mixed import robust_mahalanobis_dist_matrix, S_robust
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Shared setup — load data and CDR once

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

# ── Feature matrix ────────────────────────────────────────────────────────────
df_pca      = pd.read_csv(os.path.join(NB_DIR, "pca_all_images_merged.csv"))
patient_ids = df_pca["patient_id"].values
X           = df_pca.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)
print(f"Feature matrix X  : {X.shape}  (patients × PCA features)")
print(f"ID sample         : {patient_ids[:3].tolist()}")

# ── Clinical data ─────────────────────────────────────────────────────────────
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)

# CDR aligned to X row order
cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)  # 0→0, 0.5→1, 1→2, 2→4

print(f"Clinical table    : {df_clin.shape}")
print(f"Patients with CDR : {int(mask_cdr.sum())} / {len(patient_ids)}")
print(f"CDR categories    : {sorted(set(cdr_int))}  (CDR × 2)")

Feature matrix X  : (416, 550)  (patients × PCA features)
ID sample         : ['OAS1_0001', 'OAS1_0002', 'OAS1_0003']
Clinical table    : (416, 12)
Patients with CDR : 235 / 416
CDR categories    : [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]  (CDR × 2)


## Main experiment loop

2 metrics × 2 k values = **4 combinations**.

The Robust Mahalanobis distance matrix is precomputed once and reused for both k values.

In [4]:
METRICS  = ["euclidean", "robust_mahalanobis"]
K_VALUES = [3, 4]

# Precompute D once — reused for both k=3 and k=4
print("Precomputing Robust Mahalanobis distance matrix ...", end=" ", flush=True)
S_est     = S_robust(X, method="trimmed", alpha=0.05)
D_precomp = robust_mahalanobis_dist_matrix(X, S_est)
print(f"done  {D_precomp.shape}")

print(f"\n{'='*62}")
print(f"  pca_all_images_merged  |  X={X.shape}  |  CDR patients: {mask_cdr.sum()}")
print(f"{'='*62}")

all_results = []

for metric in METRICS:
    for k in K_VALUES:
        base = kmedoids.KMedoids(
            n_clusters=k, metric="precomputed", method="pam", random_state=42
        )
        model = SampleDistClustering(
            clustering_method=base,
            metric=metric,
            frac_sample_size=1.0,
            random_state=42,
            p1=550,
        )
        print(f"\n  [{metric}  k={k}] fitting ...", end=" ", flush=True)
        model.fit(X)
        labels = np.array(model.labels_, dtype=int)
        print("done")

        # ── % healthy per cluster (CDR = 0.0) ────────────────────────────────
        df_eval  = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
        df_eval  = df_eval[df_eval["CDR"].notnull()].copy()
        pct_tab  = pd.crosstab(
            df_eval["cluster"], df_eval["CDR"], normalize="index"
        ).mul(100).round(1)
        pct_healthy = pct_tab[0.0] if 0.0 in pct_tab.columns \
                      else pd.Series(0.0, index=pct_tab.index)
        spread = float(pct_healthy.max() - pct_healthy.min())

        # ── Silhouette ────────────────────────────────────────────────────────
        if metric == "euclidean":
            sil = silhouette_score(X, labels, metric="euclidean")
        else:
            sil = silhouette_score(D_precomp, labels, metric="precomputed")

        # ── ARI / NMI ─────────────────────────────────────────────────────────
        ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
        nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

        sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
        print(f"    sizes      : {sizes}")
        print(f"    %healthy   : { {c: round(v, 1) for c, v in pct_healthy.items()} }")
        print(f"    spread={spread:.1f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

        all_results.append({
            "metric":             metric,
            "k":                  k,
            "silhouette":         round(sil, 4),
            "ari":                round(ari, 4),
            "nmi":                round(nmi, 4),
            "spread_pct_healthy": round(spread, 2),
        })

print("\n\nAll 4 experiments complete.")

Precomputing Robust Mahalanobis distance matrix ... done  (416, 416)

  pca_all_images_merged  |  X=(416, 550)  |  CDR patients: 235

  [euclidean  k=3] fitting ... done
    sizes      : {np.int64(0): 160, np.int64(1): 102, np.int64(2): 154}
    %healthy   : {0: 53.3, 1: 58.5, 2: 63.1}
    spread=9.8%  sil=0.0371  ARI=-0.0072  NMI=0.0220

  [euclidean  k=4] fitting ... done
    sizes      : {np.int64(0): 100, np.int64(1): 80, np.int64(2): 150, np.int64(3): 86}
    %healthy   : {0: 53.8, 1: 39.7, 2: 69.1, 3: 72.7}
    spread=33.0%  sil=0.0366  ARI=0.0129  NMI=0.0498

  [robust_mahalanobis  k=3] fitting ... done
    sizes      : {np.int64(0): 262, np.int64(1): 86, np.int64(2): 68}
    %healthy   : {0: 56.6, 1: 48.0, 2: 71.4}
    spread=23.4%  sil=0.0014  ARI=-0.0006  NMI=0.0284

  [robust_mahalanobis  k=4] fitting ... done
    sizes      : {np.int64(0): 227, np.int64(1): 69, np.int64(2): 68, np.int64(3): 52}
    %healthy   : {0: 56.5, 1: 43.6, 2: 57.5, 3: 78.1}
    spread=34.5%  sil=-0.0

## Results — summary table and export

In [5]:
df_results = pd.DataFrame(all_results)

print("All results (2 metrics × 2 k = 4 rows):")
print(df_results.to_string(index=False))

out_path = os.path.join(NB_PADDED, "results_all_images_merged.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

All results (2 metrics × 2 k = 4 rows):
            metric  k  silhouette     ari    nmi  spread_pct_healthy
         euclidean  3      0.0371 -0.0072 0.0220                 9.8
         euclidean  4      0.0366  0.0129 0.0498                33.0
robust_mahalanobis  3      0.0014 -0.0006 0.0284                23.4
robust_mahalanobis  4     -0.0011 -0.0002 0.0342                34.5

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_all_images_merged.csv
